# 🧪 시나리오 3: 멀티 테넌트 토큰 할당량

## 시나리오
회사에 3개 팀이 있고, 각 팀에 다른 토큰 할당량을 부여합니다:

| 팀 | 역할 | 분당 토큰 제한 | APIM Product |
|---|---|---|---|
| 팀 A | 마케팅 | 2,000 | ai-gateway-free |
| 팀 B | 개발 | 10,000 | ai-gateway-standard |
| 팀 C | 데이터 | 50,000 | ai-gateway-premium |

## 검증 포인트
1. 팀 A가 할당량 초과 시 **429** 응답을 받는지
2. 팀 A가 차단되어도 **팀 B는 정상 호출** 가능한지 (독립 카운터)
3. 팀 C는 대량 호출도 문제없는지

## 사전 준비
APIM에서 Product와 Subscription을 먼저 설정해야 합니다.
아직 설정하지 않았다면, 이 노트북 마지막의 가이드를 참고하세요.

> 💡 Product/Subscription을 아직 만들지 않았다면, 같은 키로 `counter-key`만 다르게 설정하여 시뮬레이션할 수 있습니다.

In [ ]:
import os, time, json
import requests
from dotenv import load_dotenv

# .env에서 환경 변수 자동 로드
load_dotenv("../../.env")

APIM_URL = os.getenv("APIM_URL")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "gpt-4.1-nano")
API_VERSION = "2025-04-01-preview"

assert APIM_URL, "❌ APIM_URL이 설정되지 않았습니다. .env를 확인하세요."

# ✏️ 팀별 Subscription Key
# Product/Subscription을 설정하지 않은 경우, 동일한 키를 넣어도 됩니다.
# 이 경우 모든 팀이 같은 할당량을 공유합니다.
TEAM_KEYS = {
    "팀 A (마케팅/Free)": os.getenv("APIM_KEY_FREE", "<free-tier-subscription-key>"),
    "팀 B (개발/Standard)": os.getenv("APIM_KEY_STANDARD", "<standard-tier-subscription-key>"),
    "팀 C (데이터/Premium)": os.getenv("APIM_KEY_PREMIUM", "<premium-tier-subscription-key>"),
}

BASE_URL = f"{APIM_URL}/openai/deployments/{DEPLOYMENT_NAME}/chat/completions"

def call_as_team(team_name, prompt="Hi", max_tokens=10):
    """특정 팀의 Subscription Key로 API 호출"""
    resp = requests.post(
        BASE_URL,
        params={"api-version": API_VERSION},
        headers={
            "Content-Type": "application/json",
            "Ocp-Apim-Subscription-Key": TEAM_KEYS[team_name]
        },
        json={"messages": [{"role": "user", "content": prompt}], "max_tokens": max_tokens},
        timeout=30
    )
    return resp.status_code, resp

print("✅ 환경 설정 완료")
print(f"   APIM URL:    {APIM_URL}")
print(f"   Deployment:  {DEPLOYMENT_NAME}")
for team, key in TEAM_KEYS.items():
    masked = key[:8] + "..." if len(key) > 8 else key
    print(f"   {team}: {masked}")

## 테스트 1: 팀 A (마케팅) - 낮은 할당량 → 429 유발

분당 2,000 토큰 제한이 걸려있다면, 큰 요청을 몇 번 보내면 429가 발생해야 합니다.

**관찰 포인트:**
- `x-ratelimit-remaining-tokens` 헤더가 줄어드는 것을 확인
- 429 발생 시 `Retry-After` 헤더의 대기 시간

In [ ]:
team_a = "팀 A (마케팅/Free)"
print(f"▶ {team_a} - 토큰 할당량 소진 테스트\n")

hit_429 = False
for i in range(1, 15):
    resp = call_as_team(
        team_a,
        prompt="Write a detailed analysis of current market trends in technology sector.",
        max_tokens=500
    )
    
    remaining = resp.headers.get("x-ratelimit-remaining-tokens", "N/A")
    
    if resp.status_code == 429:
        retry_after = resp.headers.get("Retry-After", "N/A")
        print(f"  [{i:2d}] 🚫 429 Rate Limit! Retry-After: {retry_after}s")
        print(f"        응답: {resp.json().get('error', {}).get('message', resp.text)[:100]}")
        hit_429 = True
        break
    elif resp.status_code == 200:
        tokens = resp.json().get("usage", {}).get("total_tokens", "?")
        print(f"  [{i:2d}] ✅ 200  사용:{tokens} 토큰  남은:{remaining}")
    else:
        print(f"  [{i:2d}] ⚠️ {resp.status_code}: {resp.text[:100]}")
    
    time.sleep(0.2)

print()
if hit_429:
    print("✅ 팀 A가 할당량 초과로 차단되었습니다!")
else:
    print("ℹ️ 429 미발생. 할당량이 충분하거나 토큰 제한 정책이 미적용.")
    print("   → tokens-per-minute을 2000으로 낮춰보세요.")

## 테스트 2: 팀 A 차단 중, 팀 B는 정상 호출 가능한지

**핵심 검증:**  
각 팀(Subscription)의 토큰 카운터는 **독립적**이어야 합니다.  
팀 A가 429를 맞아도 팀 B는 영향을 받지 않아야 합니다.

In [ ]:
team_b = "팀 B (개발/Standard)"
print(f"▶ {team_b} - 팀 A 차단 중 독립 호출 테스트\n")

team_b_resp = call_as_team(team_b, prompt="Say hello", max_tokens=10)
team_b_status = team_b_resp.status_code

print(f"  HTTP Status: {team_b_status}")
if team_b_status == 200:
    content = team_b_resp.json()["choices"][0]["message"]["content"]
    remaining = team_b_resp.headers.get("x-ratelimit-remaining-tokens", "N/A")
    print(f"  응답: {content}")
    print(f"  남은 토큰: {remaining}")
    print()
    print("  ✅ 팀 B는 정상 호출 가능 → 할당량이 독립적으로 동작!")
elif team_b_status == 429:
    print(f"  ❌ 팀 B도 429! 할당량이 분리되지 않았습니다.")
    print(f"  → counter-key가 context.Subscription.Id 기반인지 확인하세요.")
    print(f"  → 현재 정책:")
    print(f'     counter-key="@(context.Subscription.Id)"  ← 이렇게 되어야 합니다')
else:
    print(f"  ⚠️ {team_b_resp.text[:200]}")

## 테스트 3: 팀 C (프리미엄) - 대량 호출

프리미엄 티어(분당 50,000 토큰)는 큰 요청도 여유롭게 처리해야 합니다.

In [ ]:
team_c = "팀 C (데이터/Premium)"
print(f"▶ {team_c} - 대용량 호출 테스트 (5회)\n")

success = 0
total_tokens = 0

for i in range(1, 6):
    resp = call_as_team(
        team_c,
        prompt="Analyze the impact of AI on healthcare industry.",
        max_tokens=500
    )
    
    if resp.status_code == 200:
        success += 1
        tokens = resp.json().get("usage", {}).get("total_tokens", 0)
        total_tokens += tokens
        print(f"  [{i}] ✅ 200  사용: {tokens} 토큰")
    else:
        print(f"  [{i}] ❌ {resp.status_code}")
    time.sleep(0.3)

print(f"\n  결과: {success}/5 성공, 총 {total_tokens} 토큰 사용")
if success == 5:
    print("  ✅ 프리미엄 팀은 대량 호출 문제없음!")

## 결과 요약

In [ ]:
print("═" * 55)
print(" 멀티 테넌트 토큰 할당량 테스트 요약")
print("═" * 55)
print()
print(f"  팀 A (Free):     429 발생: {'✅ Yes' if hit_429 else '❌ No'}")
print(f"  팀 B (Standard): 독립 호출: {'✅ 가능' if team_b_status == 200 else '❌ 불가'}")
print(f"  팀 C (Premium):  대량 호출: {success}/5 성공")
print()
print("─" * 55)

---

## 📌 사전 설정 가이드: Product & Subscription 만들기

### Azure Portal에서 설정

1. **Product 생성** (Azure Portal → APIM → Products → + Add)
   | Product 이름 | Display Name | Subscription Required | Published |
   |---|---|---|---|
   | ai-gateway-free | AI Gateway Free | ✅ | ✅ |
   | ai-gateway-standard | AI Gateway Standard | ✅ | ✅ |
   | ai-gateway-premium | AI Gateway Premium | ✅ | ✅ |

2. **각 Product에 API 추가** → Azure OpenAI API 선택

3. **각 Product에 정책 적용** (Product → Policies → Inbound):

   **Free (분당 2,000 토큰):**
   ```xml
   <azure-openai-token-limit
       counter-key="@(context.Subscription.Id)"
       tokens-per-minute="2000"
       estimate-prompt-tokens="true"
       remaining-tokens-header-name="x-ratelimit-remaining-tokens" />
   ```

   **Standard (분당 10,000 토큰):**
   ```xml
   <azure-openai-token-limit
       counter-key="@(context.Subscription.Id)"
       tokens-per-minute="10000"
       estimate-prompt-tokens="true"
       remaining-tokens-header-name="x-ratelimit-remaining-tokens" />
   ```

   **Premium (분당 50,000 토큰):**
   ```xml
   <azure-openai-token-limit
       counter-key="@(context.Subscription.Id)"
       tokens-per-minute="50000"
       estimate-prompt-tokens="true"
       remaining-tokens-header-name="x-ratelimit-remaining-tokens" />
   ```

4. **각 Product에 Subscription 생성** → Primary Key를 위 셀에 입력